# Lab 6 — AI Demo Day rehearsal

Ноутбук показывает зафиксированный публичный прогон пяти AI-кейсов, задержки и Plan B. Сетевые вызовы выполняет versioned runner `tests/evals/run_lab6_demo.py`; по умолчанию ноутбук только читает сохранённые доказательства и не расходует квоту Gemini.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
REPORT = ROOT / 'artifacts' / 'demo' / 'lab6-ai-demo-run.json'
report = json.loads(REPORT.read_text(encoding='utf-8'))
report['passed'], report['public_run']['health']['response']

## Сводка пяти кейсов

`stable_outcome` сравнивает продуктово значимый исход: число карточек или verdict, модель и fallback. Тексты Gemini могут различаться.

In [ ]:
rows = []
for case in report['public_run']['cases']:
    rows.append({
        'id': case['id'],
        'passed': f"{case['passed_repeats']}/{case['repeats']}",
        'stable': case['stable_outcome'],
        'median_ms': case['latency_ms']['median'],
        'p95_ms': case['latency_ms']['p95'],
        'signature': case['attempts'][0]['signature'],
    })
rows

## Проверка Plan B

In [ ]:
report['fallback_probe']['passed'], report['fallback_probe']['checks']

## Кэшированный ответ

Это статический бэкап, его нельзя выдавать за live-вызов.

In [ ]:
backup_path = ROOT / 'docs' / 'demo' / 'cache' / 'ai-demo-backup.json'
backup = json.loads(backup_path.read_text(encoding='utf-8'))
backup['warning'], [case['id'] for case in backup['cases']]

## Новый живой прогон (только перед репетицией)

Команда ниже делает 15 публичных модельных вызовов и перезаписывает evidence. Запускать осознанно из корня репозитория:

```powershell
py -3 -m tests.evals.run_lab6_demo --repeats 3 --pause-seconds 1
```